<a href="https://colab.research.google.com/github/AyaAbdElNaem/AI_Tools/blob/main/Lab5_Text_Preprocessing_Foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5: Text Preprocessing Foundations
## Why we preprocess text, what each classical method does, and when each method helps or hurts


We will go slowly and explain every important part:
1. What text preprocessing means.
2. Why raw text is hard for machines.
3. The earliest classical preprocessing methods.
4. Examples for every method.
5. What is good and what is bad about each method.
6. How to combine methods into a simple preprocessing pipeline.



## what is text preprocessing?

Text preprocessing means preparing raw text so that later NLP steps become easier, more consistent, and more useful.

Raw text is messy because language in the real world is messy:
- people use uppercase and lowercase in inconsistent ways,
- punctuation may or may not matter,
- the same idea can appear in many surface forms,
- some text contains URLs, numbers, usernames, email addresses, or repeated spaces,
- some words are very frequent but not very informative.

Preprocessing does **not** mean "remove everything".
It means "change the text carefully so the representation matches the goal of the task."

This is an important mindset:
- good preprocessing keeps useful information,
- bad preprocessing removes useful information or changes the meaning.


## one important idea (VIP):

There is no universal preprocessing pipeline that is always correct.

For example:
- In sentiment analysis, the word `not` may be extremely important.
- In document search, numbers like `2024` or `ISO 9001` may be important.
- In named entity recognition, uppercase letters may help identify names and places.
- In spam detection, URLs and punctuation may be useful signals.

So we should always ask:

**What information do I want to preserve, and what noise do I want to reduce?**


## Libraries used in this notebook

- `re`: Python's regular expression module.
  We use it for pattern-based cleaning such as removing URLs, digits, and extra spaces.
  Docs: [Python `re` documentation](https://docs.python.org/3/library/re.html)

- `string`: Python's string helper module.
  We use `string.punctuation` as a ready-made list of punctuation characters.
  Docs: [Python `string` documentation](https://docs.python.org/3/library/string.html)

- `nltk`: Natural Language Toolkit.
  A classical NLP library that provides tokenization, stopwords, stemming, and lemmatization tools.
  Docs: [NLTK documentation](https://www.nltk.org/)

- `scikit-learn`: In this notebook, we do **not** use its models.
  We only borrow the built-in English stopword list as a fallback if the NLTK stopword corpus is unavailable.
  Docs: [Scikit-learn documentation](https://scikit-learn.org/stable/documentation.html)

Specific API references used in this lab:
- [`nltk.word_tokenize`](https://www.nltk.org/api/nltk.tokenize.html)
- [`nltk.corpus.stopwords`](https://www.nltk.org/howto/corpus.html)
- [`PorterStemmer`](https://www.nltk.org/_modules/nltk/stem/porter.html)
- [`WordNetLemmatizer`](https://www.nltk.org/api/nltk.stem.wordnet.html)
- [`ENGLISH_STOP_WORDS`](https://scikit-learn.org/stable/modules/feature_extraction.html#stop-words)


## Step 1: Import libraries and prepare safe fallbacks

We will now import the libraries and define a few helper objects.

Important note:
some NLTK features depend on downloaded corpora and tokenizers.
In an offline or restricted environment, those resources may not exist yet.

So we made two things:
- it prefers the official NLTK tools when possible,
- it falls back to simpler alternatives when necessary.



In [ ]:
!pip install nltk

import nltk
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import re
import string

import pandas as pd
import nltk
from nltk.corpus import wordnet as wn
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

pd.set_option("display.max_colwidth", 120)

def nltk_resource_available(resource_path):
    try:
        nltk.data.find(resource_path)
        return True
    except LookupError:
        return False

HAS_PUNKT = nltk_resource_available("tokenizers/punkt")
HAS_STOPWORDS = nltk_resource_available("corpora/stopwords")

wn.ensure_loaded()
HAS_WORDNET = wn.synsets("dog") != []

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def safe_word_tokenize(text):
    if not text.strip():
        return []

    if HAS_PUNKT:
        return word_tokenize(text)

    return re.findall(r"\b\w+\b", text)

if HAS_STOPWORDS:
    stop_words = set(stopwords.words("english"))
    stopwords_source = "NLTK stopwords"
else:
    stop_words = set(ENGLISH_STOP_WORDS)
    stopwords_source = "scikit-learn ENGLISH_STOP_WORDS fallback"

fallback_lemma_map = {
    ("loved", "v"): "love",
    ("loving", "v"): "love",
    ("hated", "v"): "hate",
    ("hating", "v"): "hate",
    ("studies", "v"): "study",
    ("studying", "v"): "study",
    ("running", "v"): "run",
    ("better", "a"): "good",
    ("cars", "n"): "car",
    ("mice", "n"): "mouse",
    ("children", "n"): "child",
}

def safe_lemmatize(token, pos="n"):
    token = token.lower()

    if HAS_WORDNET:
        return lemmatizer.lemmatize(token, pos=pos)

    if (token, pos) in fallback_lemma_map:
        return fallback_lemma_map[(token, pos)]

    if pos == "n":
        if token.endswith("ies") and len(token) > 3:
            return token[:-3] + "y"
        if token.endswith("s") and not token.endswith("ss") and len(token) > 3:
            return token[:-1]

    if pos == "v":
        if token.endswith("ing") and len(token) > 4:
            base = token[:-3]
            if len(base) >= 2 and base[-1] == base[-2]:
                base = base[:-1]
            if base.endswith(("v", "t")):
                return base + "e"
            return base

        if token.endswith("ed") and len(token) > 3:
            base = token[:-2]
            if base.endswith(("v", "t")):
                return base + "e"
            return base

    return token

resource_status_df = pd.DataFrame(
    {
        "resource_or_fallback": [
            "punkt tokenizer",
            "stopwords list",
            "wordnet lemmatizer",
        ],
        "available": [HAS_PUNKT, HAS_STOPWORDS, HAS_WORDNET],
        "behavior_if_missing": [
            "Use regex tokenization fallback",
            "Use scikit-learn stopwords fallback",
            "Use a small rule-based teaching fallback",
        ],
    }
)

resource_status_df


,resource_or_fallback,available,behavior_if_missing
0,punkt tokenizer,True,Use regex tokenization fallback
1,stopwords list,True,Use scikit-learn stopwords fallback
2,wordnet lemmatizer,True,Use a small rule-based teaching fallback


## Step 2: Start with messy raw text

We will use a small set of examples that contain several common types of noise:
- repeated punctuation,
- extra spaces,
- uppercase letters,
- numbers,
- URLs,
- contractions,
- words whose meaning could be damaged by aggressive cleaning.


In [ ]:
raw_examples = [
    "I LOVE this movie!!!",
    "This movie was not bad at all :)",
    "Visit https://openai.com/docs right now!!!",
    "Python     is     very     fun.",
    "The price is 199.99 dollars.",
    "Call me at 01234 567890 tomorrow.",
    "US policy changed in May 2024.",
    "I can't say this product is amazing."
]

raw_df = pd.DataFrame({"raw_text": raw_examples})
raw_df


,raw_text
0,I LOVE this movie!!!
1,This movie was not bad at all :)
2,Visit https://openai.com/docs right now!!!
3,Python is very fun.
4,The price is 199.99 dollars.
5,Call me at 01234 567890 tomorrow.
6,US policy changed in May 2024.
7,I can't say this product is amazing.


## Why raw text causes problems

Humans can see that many of these examples are easy to understand.
Machines do not "understand" them the same way.

A machine initially sees text as strings and tokens.
Small differences in surface form can create different representations:
- `LOVE` vs `love`
- `movie!!!` vs `movie`
- `199.99` vs `199`
- `can't` vs `cant`

Preprocessing exists to reduce unnecessary variation.
But reducing variation too aggressively can also remove meaning.


In [ ]:
sentence_a = "I LOVE this movie!!!"
sentence_b = "i love this movie"

def whitespace_tokenize(text):
    return text.split()

print("Sentence A using split():", whitespace_tokenize(sentence_a))
print("Sentence B using split():", whitespace_tokenize(sentence_b))


Sentence A using split(): ['I', 'LOVE', 'this', 'movie!!!']
Sentence B using split(): ['i', 'love', 'this', 'movie']


Even in this tiny example, we see the problem:

- `LOVE` and `love` are different strings,
- `movie!!!` and `movie` are different tokens,
- simple splitting leaves punctuation attached to words.

So the big goal of preprocessing is not "beauty".
The goal is **consistency**.


## A useful mental model: preprocessing categories

We can loosely divide preprocessing into three categories:

1. **Cleaning**
   Remove obvious noise such as repeated spaces or URLs.

2. **Normalization**
   Convert different surface forms into more consistent forms, such as `LOVE` -> `love`.

3. **Segmentation**
   Break text into smaller units such as tokens.

Later labs can go deeper into feature extraction and model behavior.
For now, we stay focused on these early classical steps.


## Method 1: Lowercasing

Lowercasing converts all letters to lowercase.

Why people use it:
- It reduces duplicate forms like `Movie`, `movie`, and `MOVIE`.
- It makes the vocabulary smaller.
- It is simple and usually useful in classical NLP pipelines.

What is good about lowercasing:
- Very easy to apply.
- Usually improves consistency.

What is bad about lowercasing:
- It removes case information.
- That can hurt tasks where case matters:
  - `US` vs `us`
  - `May` the month vs `may` the verb
  - named entities like `Apple` the company vs `apple` the fruit


In [ ]:
lower_examples = [
    "I LOVE NLP",
    "US policy changed",
    "May is a warm month",
    "Apple released a new device"
]

lower_df = pd.DataFrame(
    {
        "original": lower_examples,
        "lowercased": [text.lower() for text in lower_examples],
    }
)

lower_df


,original,lowercased
0,I LOVE NLP,i love nlp
1,US policy changed,us policy changed
2,May is a warm month,may is a warm month
3,Apple released a new device,apple released a new device


Notice the tradeoff:

- `I LOVE NLP` -> useful simplification.
- `US policy changed` -> maybe harmful, because `US` becomes `us`.
- `May` -> maybe harmful, because the month loses its capitalized clue.

So lowercasing is often a good default for simple text classification pipelines, but not always a good default for tasks that care about named entities or exact forms.


## Method 2: Punctuation removal

Punctuation removal deletes characters such as `. , ! ? : ; ' " ( )`.

Why people use it:
- Punctuation often creates noisy token variants like `movie!!!`.
- Classical count-based models often do not benefit much from punctuation.

What is good about punctuation removal:
- Makes tokens cleaner.
- Reduces variation.
- Very common in simple pipelines.

What is bad about punctuation removal:
- Punctuation can carry meaning.
- `can't` becomes `cant`.
- `C++` may become `C`.
- `10/10` may become `1010`.
- emoticons like `:)` may disappear completely.

So punctuation removal is helpful when punctuation is mostly noise, but risky when punctuation contributes meaning.


In [ ]:
translator = str.maketrans("", "", string.punctuation)

def remove_punctuation(text):
    return text.translate(translator)

punctuation_examples = [
    "Hello, world!",
    "I can't wait.",
    "C++ is powerful.",
    "This is great :)",
    "I rate it 10/10."
]

punctuation_df = pd.DataFrame(
    {
        "original": punctuation_examples,
        "without_punctuation": [remove_punctuation(text) for text in punctuation_examples],
    }
)

punctuation_df


,original,without_punctuation
0,"Hello, world!",Hello world
1,I can't wait.,I cant wait
2,C++ is powerful.,C is powerful
3,This is great :),This is great
4,I rate it 10/10.,I rate it 1010


A few observations:

- `Hello, world!` becomes cleaner.
- `can't` becomes `cant`, which changes the form.
- `C++` becomes `C`, which is usually too destructive.
- `:)` disappears, which may remove sentiment information.

This is a good example of why preprocessing should be chosen with care.


## Method 3: Number handling

Number handling means deciding what to do with digits:
- keep them,
- remove them,
- replace them with a placeholder,
- or normalize them in another way.

Why people sometimes remove numbers:
- In some text tasks, many numbers are just noise.
- Numbers can create many rare token variants.

What is good about removing numbers:
- Simpler text.
- Smaller vocabulary.
- Sometimes less noise.

What is bad about removing numbers:
- Numbers can be very important.
- Prices, dates, versions, phone numbers, IDs, ratings, and quantities may matter a lot.
- `199.99 dollars` and `10/10` carry useful meaning in many tasks.

So number removal should almost never be automatic without thinking about the task.


In [ ]:
def remove_numbers(text):
    return re.sub(r"\d+", "", text)

number_examples = [
    "The price is 199.99 dollars.",
    "I rate it 10/10.",
    "US policy changed in 2024.",
    "Model version is 3.5."
]

number_df = pd.DataFrame(
    {
        "original": number_examples,
        "after_number_removal": [remove_numbers(text) for text in number_examples],
    }
)

number_df


,original,after_number_removal
0,The price is 199.99 dollars.,The price is . dollars.
1,I rate it 10/10.,I rate it /.
2,US policy changed in 2024.,US policy changed in .
3,Model version is 3.5.,Model version is ..


We can see the downside clearly:

- `199.99` loses the amount.
- `10/10` becomes distorted.
- `2024` disappears even if the year matters.
- version numbers may become incomplete.

In many real systems, replacing numbers with a tag like `<NUM>` can be better than deleting them.
We are not doing that yet, but it is a useful future idea.


## Method 4: Whitespace normalization

Whitespace normalization means turning repeated spaces, tabs, or line breaks into a simpler consistent form.

Why people use it:
- Extra spaces usually do not carry meaning.
- Inconsistent spacing can make text harder to inspect and tokenize.

What is good about whitespace normalization:
- Safe in many tasks.
- Easy to apply.
- Improves readability and consistency.

What is bad about whitespace normalization:
- In a few special tasks, layout and spacing might matter.
- For example, some OCR, document-layout, or code-related tasks may care about original spacing.


In [ ]:
def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

whitespace_examples = [
    "Python     is     very     fun.",
    "Line one\n\nLine two\t\tLine three",
    "   Leading and trailing spaces   "
]

whitespace_df = pd.DataFrame(
    {
        "original": whitespace_examples,
        "normalized": [normalize_whitespace(text) for text in whitespace_examples],
    }
)

whitespace_df


,original,normalized
0,Python is very fun.,Python is very fun.
1,Line one\n\nLine two\t\tLine three,Line one Line two Line three
2,Leading and trailing spaces,Leading and trailing spaces


Among classical preprocessing steps, whitespace normalization is one of the safest and easiest.

It usually helps more than it hurts, especially in beginner pipelines.


## Method 5: URL removal

Many real text datasets contain links:
- `https://...`
- `http://...`
- `www....`

Why people remove URLs:
- Links are often noisy for general text understanding.
- Every unique URL can become a strange rare token.

What is good about URL removal:
- Removes obvious noise.
- Reduces unique messy tokens.

What is bad about URL removal:
- Sometimes the presence of a URL is informative.
- In spam detection, fraud detection, or social media analysis, URLs may matter.
- The domain itself may carry useful information.

So removing URLs is common, but sometimes replacing them with a placeholder like `<URL>` is better than deleting them.


In [ ]:
def remove_urls(text):
    return re.sub(r"http\S+|www\.\S+", "", text)

url_examples = [
    "Visit https://openai.com/docs right now!!!",
    "Read more at www.example.org/news.",
    "This message has no link."
]

url_df = pd.DataFrame(
    {
        "original": url_examples,
        "after_url_removal": [normalize_whitespace(remove_urls(text)) for text in url_examples],
    }
)

url_df


,original,after_url_removal
0,Visit https://openai.com/docs right now!!!,Visit right now!!!
1,Read more at www.example.org/news.,Read more at
2,This message has no link.,This message has no link.


URL removal is often helpful in first-pass cleaning.

But if you later work on tasks involving trust, spam, source identity, or domain behavior, you may want to keep the URL or replace it with a meaningful placeholder instead of deleting it.


## Method 6: Tokenization

Tokenization means splitting text into smaller units called tokens.
In simple pipelines, tokens are often words.

Why tokenization matters:
- Most later NLP steps work on tokens, not raw full strings.
- Stopword removal, stemming, and lemmatization usually happen after tokenization.

We will compare two simple approaches:
- `text.split()`
- `safe_word_tokenize()`

`safe_word_tokenize()` uses NLTK's `word_tokenize()` when possible and a regex fallback otherwise.

What is good about `split()`:
- Very simple.
- Good for first intuition.

What is bad about `split()`:
- It only splits on spaces.
- Punctuation stays attached to words.
- It does not reflect language-specific tokenization rules.

What is good about `word_tokenize()`:
- More NLP-aware than a plain space split.
- Better at handling punctuation boundaries.

What is bad about tokenization in general:
- No tokenizer is perfect for every domain.
- Contractions, hashtags, emojis, and mixed-language text can be tricky.


In [ ]:
tokenization_examples = [
    "NLP is fun, powerful, and practical.",
    "I can't do this.",
    "State-of-the-art systems are impressive."
]

tokenization_rows = []

for text in tokenization_examples:
    tokenization_rows.append(
        {
            "text": text,
            "split_tokens": text.split(),
            "safe_word_tokenize_tokens": safe_word_tokenize(text),
        }
    )

tokenization_df = pd.DataFrame(tokenization_rows)
tokenization_df


,text,split_tokens,safe_word_tokenize_tokens
0,"NLP is fun, powerful, and practical.","[NLP, is, fun,, powerful,, and, practical.]","[NLP, is, fun, ,, powerful, ,, and, practical, .]"
1,I can't do this.,"[I, can't, do, this.]","[I, ca, n't, do, this, .]"
2,State-of-the-art systems are impressive.,"[State-of-the-art, systems, are, impressive.]","[State-of-the-art, systems, are, impressive, .]"


Look carefully at the tokens:

- `split()` usually leaves commas and periods attached.
- More NLP-aware tokenization usually separates punctuation more clearly.
- Contractions and hyphenated words remain challenging.

This shows an important truth:
tokenization is not a trivial detail.
It shapes the units that every later preprocessing step will operate on.


## Method 7: Stopword removal

Stopwords are very frequent words such as:
- `the`
- `is`
- `and`
- `of`

Why people remove stopwords:
- These words often appear in many sentences.
- In simple count-based pipelines, they may contribute less signal than content words.

What is good about stopword removal:
- Can reduce text size.
- Can make content words more prominent.
- Often useful in search and classical bag-of-words pipelines.

What is bad about stopword removal:
- Some so-called stopwords are actually important.
- `not`, `no`, `never`, and similar words may carry crucial meaning.
- In some tasks, grammatical words matter.

So stopword removal should be selective, not blind.


In [ ]:
print("Stopword source in this notebook:", stopwords_source)

stopword_example = "This is not a good movie and I do not like it at all"
stopword_tokens = safe_word_tokenize(stopword_example.lower())

tokens_without_stopwords = [
    token for token in stopword_tokens if token not in stop_words
]

protected_negation_words = {"no", "not", "nor", "never"}
tokens_without_stopwords_but_keep_negation = [
    token
    for token in stopword_tokens
    if (token not in stop_words) or (token in protected_negation_words)
]

stopword_df = pd.DataFrame(
    {
        "version": [
            "original tokens",
            "after plain stopword removal",
            "after stopword removal while keeping negation words",
        ],
        "tokens": [
            stopword_tokens,
            tokens_without_stopwords,
            tokens_without_stopwords_but_keep_negation,
        ],
    }
)

stopword_df


Stopword source in this notebook: NLTK stopwords


,version,tokens
0,original tokens,"[this, is, not, a, good, movie, and, i, do, not, like, it, at, all]"
1,after plain stopword removal,"[good, movie, like]"
2,after stopword removal while keeping negation words,"[not, good, movie, not, like]"


This is one of the best examples of preprocessing risk:

- If we remove `not`, the sentence may look much more positive than it really is.
- If we keep negation words, we preserve more meaning.

So the "good" version depends on the task.
For sentiment and stance tasks, keeping negation is often a better choice.


## Method 8: Stemming

Stemming tries to reduce related word forms to a common root or stem.

Example family:
- `connect`
- `connected`
- `connecting`
- `connection`

Why people use stemming:
- It groups related forms together.
- It reduces vocabulary size.
- It is computationally simple.

We will use `PorterStemmer`, one of the most classical stemmers.

What is good about stemming:
- Fast.
- Useful for simple classical NLP pipelines.
- Often enough when exact readability is not important.

What is bad about stemming:
- The stem may not be a real dictionary word.
- Different words may be reduced too aggressively.
- Outputs can be hard for humans to read.

Stemming is often useful for classical information retrieval and simple text normalization, but not always ideal when we care about clean human-readable output.


In [ ]:
stemming_words = [
    "connect",
    "connected",
    "connection",
    "connecting",
    "studies",
    "better",
    "generously"
]

stemming_df = pd.DataFrame(
    {
        "original_word": stemming_words,
        "stemmed_word": [stemmer.stem(word) for word in stemming_words],
    }
)

stemming_df


,original_word,stemmed_word
0,connect,connect
1,connected,connect
2,connection,connect
3,connecting,connect
4,studies,studi
5,better,better
6,generously,gener


Notice:

- Some outputs look reasonable.
- Some outputs are rough and not very readable.
- The goal of stemming is consistency, not beauty.

This is why stemming is often described as a rough normalization method.


## Method 9: Lemmatization

Lemmatization also reduces words to a base form, but it tries to return a real dictionary form called the lemma.

Examples:
- `loved` -> `love`
- `cars` -> `car`
- `mice` -> `mouse`

Why people use lemmatization:
- It usually produces cleaner and more interpretable forms than stemming.
- It is often easier to explain to humans.

What is good about lemmatization:
- More readable outputs.
- Often linguistically cleaner.
- Better when interpretability matters.

What is bad about lemmatization:
- It is more resource-dependent than stemming.
- Correct results often depend on part-of-speech information.
- It can be slower or more complex than stemming.

In this notebook, we will show noun and verb behavior because the result can change with part of speech.


In [ ]:
lemmatization_words = ["loved", "loving", "studies", "studying", "cars", "mice", "children"]

lemmatization_df = pd.DataFrame(
    {
        "original_word": lemmatization_words,
        "lemma_as_noun": [safe_lemmatize(word, pos="n") for word in lemmatization_words],
        "lemma_as_verb": [safe_lemmatize(word, pos="v") for word in lemmatization_words],
    }
)

lemmatization_df


,original_word,lemma_as_noun,lemma_as_verb
0,loved,loved,love
1,loving,loving,love
2,studies,study,study
3,studying,studying,study
4,cars,car,cars
5,mice,mouse,mice
6,children,child,children


This is an important lesson:

- Lemmatization is not just "remove endings".
- The intended grammatical role matters.
- The same word may produce different lemmas depending on whether we treat it as a noun or a verb.

That makes lemmatization more linguistically meaningful, but also a little more complex than stemming.


## Stemming vs lemmatization

Both methods aim to reduce variation, but they do it differently.

Stemming:
- simpler,
- rougher,
- faster,
- less readable.

Lemmatization:
- cleaner,
- more interpretable,
- more linguistically informed,
- often more resource-dependent.

Let us compare them directly.


In [ ]:
comparison_words = ["loved", "loving", "studies", "studying", "running", "better", "children"]

stem_vs_lemma_df = pd.DataFrame(
    {
        "word": comparison_words,
        "stem": [stemmer.stem(word) for word in comparison_words],
        "lemma_as_noun": [safe_lemmatize(word, pos="n") for word in comparison_words],
        "lemma_as_verb": [safe_lemmatize(word, pos="v") for word in comparison_words],
    }
)

stem_vs_lemma_df


,word,stem,lemma_as_noun,lemma_as_verb
0,loved,love,loved,love
1,loving,love,loving,love
2,studies,studi,study,study
3,studying,studi,studying,study
4,running,run,running,run
5,better,better,better,better
6,children,children,child,children


A useful rule of thumb:

- If you want a very classical and compact normalization step, stemming may be enough.
- If you want cleaner human-readable normalized words, lemmatization is often the better choice.

But the best choice still depends on the task, data, language, and available tools.


## Why the order of preprocessing steps matters

The order of operations can change the output.

Example:
- If we remove stopwords before lowercasing, `This` may not match the lowercase stopword `this`.
- If we remove punctuation before thinking about contractions, `can't` becomes `cant`.
- If we remove URLs after other modifications, detection may become harder.

So a pipeline is not just a list of steps.
It is also a chosen order.


In [ ]:
order_example = "This is NOT a good movie!!!"

lower_then_stop = [
    token
    for token in safe_word_tokenize(order_example.lower())
    if token not in stop_words
]

stop_then_lower = [
    token.lower()
    for token in safe_word_tokenize(order_example)
    if token not in stop_words
]

order_df = pd.DataFrame(
    {
        "pipeline_order": [
            "lowercase first, then remove stopwords",
            "remove stopwords first, then lowercase",
        ],
        "tokens": [lower_then_stop, stop_then_lower],
    }
)

order_df


,pipeline_order,tokens
0,"lowercase first, then remove stopwords","[good, movie, !, !, !]"
1,"remove stopwords first, then lowercase","[this, not, good, movie, !, !, !]"


In practice, lowercasing before stopword removal is often more consistent because stopword lists are usually lowercase.


## Build one simple reusable preprocessing pipeline

Now we combine the earlier ideas into one flexible function.

This function can:
- lowercase text,
- remove URLs,
- remove punctuation,
- remove numbers,
- normalize spaces,
- tokenize,
- remove stopwords,
- optionally preserve negation words,
- optionally apply stemming,
- optionally apply lemmatization.

We will keep it explicit and readable, because the goal is teaching.


In [ ]:
def preprocess_text(
    text,
    lowercase=True,
    remove_url=True,
    remove_punct=True,
    remove_num=False,
    normalize_space=True,
    remove_stop_words=False,
    preserve_negation=True,
    use_stemming=False,
    use_lemmatization=False,
):
    if use_stemming and use_lemmatization:
        raise ValueError("Use either stemming or lemmatization, not both at the same time.")

    if lowercase:
        text = text.lower()

    if remove_url:
        text = remove_urls(text)

    if remove_punct:
        text = remove_punctuation(text)

    if remove_num:
        text = remove_numbers(text)

    if normalize_space:
        text = normalize_whitespace(text)

    if not text:
        return ""

    tokens = safe_word_tokenize(text)

    if remove_stop_words:
        if preserve_negation:
            protected_negation_words = {"no", "not", "nor", "never"}
            tokens = [token for token in tokens if (token not in stop_words) or (token in protected_negation_words)]
        else:
            tokens = [token for token in tokens if token not in stop_words]

    if use_stemming:
        tokens = [stemmer.stem(token) for token in tokens]

    if use_lemmatization:
        tokens = [safe_lemmatize(token, pos="v") for token in tokens]

    return " ".join(tokens)


## Try different preprocessing profiles on the same texts

Instead of claiming that one pipeline is always best, we will compare several styles:

- `minimal_clean`:
  safe basic cleaning with no stopword removal and no stemming.

- `stopword_reduced`:
  remove stopwords but keep negation words.

- `aggressive_stemmed`:
  remove stopwords, remove numbers, and stem aggressively.

- `readable_lemmatized`:
  remove stopwords, keep negation, and use lemmatization for cleaner output.

This helps us see how design choices affect the final text.


In [ ]:
preprocessing_profiles = {
    "minimal_clean": dict(
        lowercase=True,
        remove_url=True,
        remove_punct=True,
        remove_num=False,
        normalize_space=True,
        remove_stop_words=False,
        preserve_negation=True,
        use_stemming=False,
        use_lemmatization=False,
    ),
    "stopword_reduced": dict(
        lowercase=True,
        remove_url=True,
        remove_punct=True,
        remove_num=False,
        normalize_space=True,
        remove_stop_words=True,
        preserve_negation=True,
        use_stemming=False,
        use_lemmatization=False,
    ),
    "aggressive_stemmed": dict(
        lowercase=True,
        remove_url=True,
        remove_punct=True,
        remove_num=True,
        normalize_space=True,
        remove_stop_words=True,
        preserve_negation=False,
        use_stemming=True,
        use_lemmatization=False,
    ),
    "readable_lemmatized": dict(
        lowercase=True,
        remove_url=True,
        remove_punct=True,
        remove_num=False,
        normalize_space=True,
        remove_stop_words=True,
        preserve_negation=True,
        use_stemming=False,
        use_lemmatization=True,
    ),
}

profile_rows = []

for text in raw_examples:
    row = {"raw_text": text}
    for profile_name, profile_kwargs in preprocessing_profiles.items():
        row[profile_name] = preprocess_text(text, **profile_kwargs)
    profile_rows.append(row)

profiles_df = pd.DataFrame(profile_rows)
profiles_df


,raw_text,minimal_clean,stopword_reduced,aggressive_stemmed,readable_lemmatized
0,I LOVE this movie!!!,i love this movie,love movie,love movi,love movie
1,This movie was not bad at all :),this movie was not bad at all,movie not bad,movi bad,movie not bad
2,Visit https://openai.com/docs right now!!!,visit right now,visit right,visit right,visit right
3,Python is very fun.,python is very fun,python fun,python fun,python fun
4,The price is 199.99 dollars.,the price is 19999 dollars,price 19999 dollars,price dollar,price 19999 dollars
5,Call me at 01234 567890 tomorrow.,call me at 01234 567890 tomorrow,call 01234 567890 tomorrow,call tomorrow,call 01234 567890 tomorrow
6,US policy changed in May 2024.,us policy changed in may 2024,us policy changed may 2024,us polici chang may,us policy change may 2024
7,I can't say this product is amazing.,i cant say this product is amazing,cant say product amazing,cant say product amaz,cant say product amaze


This table is one of the most important outputs in the notebook.

It shows that preprocessing is really a set of tradeoffs:
- More aggressive cleaning may create more consistency.
- But more aggressive cleaning may also remove useful meaning.
- Readability for humans is often better with lemmatization than with stemming.
- Keeping numbers or negation words can preserve important information.


## A practical decision guide for each method

The table below summarizes when each technique is often helpful and when it may be risky.


In [ ]:
decision_guide_df = pd.DataFrame(
    {
        "method": [
            "Lowercasing",
            "Punctuation removal",
            "Number removal",
            "Whitespace normalization",
            "URL removal",
            "Stopword removal",
            "Stemming",
            "Lemmatization",
        ],
        "often_helpful_when": [
            "You want a smaller, more consistent vocabulary",
            "Punctuation is mostly noise",
            "Numbers are unimportant or too noisy",
            "Spacing is inconsistent and not meaningful",
            "Links are mostly noise",
            "Function words add little value",
            "You want rough fast normalization",
            "You want cleaner dictionary-like base forms",
        ],
        "risky_when": [
            "Case carries meaning such as names or abbreviations",
            "Punctuation changes meaning such as can't, C++, 10/10, emoticons",
            "Dates, prices, ratings, IDs, or versions matter",
            "Layout or formatting carries meaning",
            "Links or domains are informative",
            "Negation or grammar matters",
            "Human readability matters or over-reduction is harmful",
            "You lack the right linguistic resources or part-of-speech information",
        ],
    }
)

decision_guide_df


,method,often_helpful_when,risky_when
0,Lowercasing,"You want a smaller, more consistent vocabulary",Case carries meaning such as names or abbreviations
1,Punctuation removal,Punctuation is mostly noise,"Punctuation changes meaning such as can't, C++, 10/10, emoticons"
2,Number removal,Numbers are unimportant or too noisy,"Dates, prices, ratings, IDs, or versions matter"
3,Whitespace normalization,Spacing is inconsistent and not meaningful,Layout or formatting carries meaning
4,URL removal,Links are mostly noise,Links or domains are informative
5,Stopword removal,Function words add little value,Negation or grammar matters
6,Stemming,You want rough fast normalization,Human readability matters or over-reduction is harmful
7,Lemmatization,You want cleaner dictionary-like base forms,You lack the right linguistic resources or part-of-speech information


## What often get wrong

These are very common mistakes:

1. Removing everything by default.
   More cleaning is not always better.

2. Removing negation words.
   `not good` is not the same as `good`.

3. Removing numbers without checking whether they matter.
   Ratings, prices, and years are often important.

4. Assuming stemming and lemmatization are the same.
   They are related, but they produce different outputs and serve slightly different needs.

5. Forgetting that preprocessing order matters.
   The same steps in a different order may produce different results.


## Recommended mindset

Start simple:
- normalize whitespace,
- decide whether to lowercase,
- decide whether URLs and punctuation are noise,
- tokenize carefully,
- remove stopwords only if it helps the task,
- choose stemming or lemmatization only when you have a reason.

In other words:

**Do not preprocess because it is traditional. Preprocess because it fits the task.**


## Lab 5 summary

In this lab we stayed entirely focused on preprocessing itself.

We covered:
- why preprocessing exists,
- why raw text is hard for machines,
- lowercasing,
- punctuation removal,
- number handling,
- whitespace normalization,
- URL removal,
- tokenization,
- stopword removal,
- stemming,
- lemmatization,
- the importance of step ordering,
- how to build a simple reusable preprocessing pipeline.

Main takeaway:

Text preprocessing is the art of reducing unnecessary variation **without deleting important meaning**.


## Practice tasks

Try these exercises:

1. Add a sentence that contains a product code such as `RTX 4090` and see whether number removal still looks like a good idea.
2. Add a sentence with a country name or person name and inspect whether lowercasing removes useful information.
3. Modify `preprocess_text()` so URLs are replaced with `<URL>` instead of being deleted.
4. Compare the outputs of stopword removal with `preserve_negation=True` and `False`.
5. Add your own words to compare stemming and lemmatization.
6. Create one preprocessing profile for sentiment analysis and another for document search, then compare them.
7. we will provide you data to build different preprocessing profiles then i will test your profiles with another data
